# JP-RU Historical Terms: Интеллектуальный исторический глоссарий для рабочего места переводчика-япониста


In [2]:
### 1. Установка и импорт
import time, re, logging, warnings, wikipedia
warnings.filterwarnings('ignore')
logging.getLogger("transformers").setLevel(logging.ERROR)
import requests
import time
import random
import json                      
from typing import Dict, Optional
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
print("Библиотеки успешно загружены")

### 2. Мониторинг

class AgentMonitor:
    def __init__(self):
        self.reset()
    def reset(self):
        self.start = self.end = time.time()
        self.llm_calls = self.wiki_calls = 0
    def print_metrics(self):
        print(f"\n{'='*50}\nОтчёт JP→RU агента\n{'='*50}")
        print(f"Время: {self.end-self.start:.2f} с")
        print(f"LLM-запросы: {self.llm_calls}, Wiki-запросы: {self.wiki_calls}\n{'='*50}\n")

monitor = AgentMonitor()
print("Мониторинг готов")

### 3. Загрузка компактной японско-русской модели

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype="auto", device_map="auto")
pipe = pipeline("text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=300,
                temperature=0.05,
                return_full_text=False,
                pad_token_id=tokenizer.eos_token_id)
llm = HuggingFacePipeline(pipeline=pipe)
print(f"Модель {model_id} загружена и готова к работе")


Библиотеки успешно загружены
Мониторинг готов


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


Модель Qwen/Qwen2.5-1.5B-Instruct загружена и готова к работе


C:\Users\user\AppData\Local\Temp\ipykernel_51880\2806433588.py:43: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [3]:
### 4. Утилиты
# ------------------------------------------------------------------
# Утилита: QID + sitelinks из Wikidata
# ------------------------------------------------------------------
HEADERS = {"User-Agent": "ja2ru-helper/0.1 (https://github.com/jhgudleik/)"} 

def get_qid_sitelinks(page_title: str, lang: str = "ja") -> Dict[str, str]:
    """
    Возвращает словарь {язык: заголовок} для статьи page_title в Википедии языка lang.
    Если статья не привязана к Викиданным — вернёт {}.
    """
    session = requests.Session()
    session.headers.update(HEADERS)

    # 1) Получаем QID
    wiki_api = f"https://{lang}.wikipedia.org/w/api.php"
    try:
        r = session.get(
            wiki_api,
            params={
                "action": "query",
                "prop": "pageprops",
                "ppprop": "wikibase_item",
                "titles": page_title,
                "format": "json"
            },
            timeout=10
        )
        r.raise_for_status()
        data = r.json()
    except (requests.exceptions.RequestException, json.JSONDecodeError) as e:  # ← исправлено
        print(f"⚠️ Ошибка QID для «{page_title}»: {e}")
        return {}

    pages = data.get("query", {}).get("pages", {})
    qid = next(iter(pages.values()), {}).get("pageprops", {}).get("wikibase_item")
    if not qid:
        return {}

    # 2) По QID забираем sitelinks
    try:
        r2 = session.get(
            "https://www.wikidata.org/w/api.php",
            params={
                "action": "wbgetentities",
                "ids": qid,
                "props": "sitelinks",
                "format": "json"
            },
            timeout=10
        )
        r2.raise_for_status()
        data2 = r2.json()
    except (requests.exceptions.RequestException, json.JSONDecodeError) as e:  # ← исправлено
        print(f"⚠️ Ошибка sitelinks для {qid}: {e}")
        return {}

    sitelinks = data2.get("entities", {}).get(qid, {}).get("sitelinks", {})
    return {s.replace("wiki", ""): data["title"] for s, data in sitelinks.items() if s.endswith("wiki")}


# ------------------------------------------------------------------
# Утилита ja2ru_wiki_page
# ------------------------------------------------------------------
def ja2ru_wiki_page(ja_term: str):
    """Возвращает рус_заголовок, рус_текст по японскому термину."""
    monitor.wiki_calls += 1
    wikipedia.set_lang("ja")
    try:
        ja_page = wikipedia.page(ja_term, auto_suggest=False)
    except Exception as e:
        print("❌ Японская страница не найдена:", e)
        return None, None

    # --- новый код вместо ja_page.langlinks -------------------------
    sitelinks = get_qid_sitelinks(ja_page.title, lang="ja")
    ru_title = sitelinks.get("ru")
    if not ru_title:
        print("❌ Русская интер-вики ссылка не найдена")
        return None, None
    # ---------------------------------------------------------------

    wikipedia.set_lang("ru")
    ru_page = wikipedia.page(ru_title, auto_suggest=False)
    return ru_page.title, ru_page.content[:1_500]


### 5. Ядро агента
def translate_ja_term(ja_query: str):
    print(f"ЗАПРОС ПЕРЕВОДЧИКА: {ja_query}\n")
    monitor.reset()
    monitor.start = time.time()

    # --- 1. Извлекаем японский термин
    monitor.llm_calls += 1
    '''
    extract_prompt = f"""<|im_start|>system
Выдели из текста японский термин в смешанной японской записи и ответь только этим словом, без пояснений.
<|im_start|>user
{ja_query}<|im_start|>assistant"""
    '''


    extract_prompt = f"""<|im_start|>system
Ты извлекаешь японский термин из текста пользователя.
1. Ответь **точно теми же иероглифами / каной**, что есть во входе.
2. Не переводи, не транскрибируй, не добавляй пояснений.
3. Выведи **одну непрерывную строку** только из японских символов (кандзи, хирагана, катакана).
4. Если таких символов несколько — верни самую длинную непрерывную последовательность.

Примеры:
Вход: 船手頭是什么？ → 船手頭
Вход: あの人は戦国時代が好きだ → 戦国時代
<|im_start|>user
{ja_query}
<|im_start|>assistant"""

    term_ja = pipe(extract_prompt)[0]["generated_text"].strip()

    print("Найден термин:", term_ja)

    # --- 2. Ищем в ja-Wiki + берём ru-статью
    ru_title, ru_text = ja2ru_wiki_page(term_ja)

    if not ru_title:
        return "Русская статья не найдена, попробуйте переформулировать."

    print("Найдена русская статья:", ru_title)
    print("Текст статьи (первые 300 символов):", ru_text[:300])

    # --- 3. Генерируем перевод и справку
    monitor.llm_calls += 1
    final_prompt = f"""<|im_start|>system
Ты — помощник переводчика с японского на русский. Используя факты из русской Википедии (ниже), дай:
1) сообщи, что самый удачный вариант перевода на русский это - заголовок аналогичной статьи в русской Википедии: {ru_title};
2) дай дословный перевод японского термина {term_ja} на русский;
3) дай краткую справку (1-2 предложения) о том, что это такое, используя факты из статьи.

Факты из Википедии:
{ru_text}
<|im_start|>user
Как перевести «{term_ja}»?<|im_start|>assistant"""
    answer = pipe(final_prompt)[0]["generated_text"].strip()

    monitor.end = time.time()
    monitor.print_metrics()
    print(f"СПРАВКА ДЛЯ ПЕРЕВОДЧИКА:\n")
    return answer

### 6. Демо-запуск
if __name__ == "__main__":
    sample = "В тексте встретилось 江戸幕府. Как правильно перевести?"
    print(translate_ja_term(sample))



ЗАПРОС ПЕРЕВОДЧИКА: В тексте встретилось 江戸幕府. Как правильно перевести?

Найден термин: 江戸幕府
Найдена русская статья: Сёгунат Токугава
Текст статьи (первые 300 символов): Сёгуна́т Токуга́ва (яп. 徳川幕府 токугава бакуфу), или Сёгуна́т Э́до (яп. 江戸幕府 эдо бакуфу), также Токугавская Япония— феодальное военное правительство Японии, основанное в 1603 году Токугавой Иэясу и возглавляемое сёгунами из рода Токугава. Просуществовало более двух с половиной веков вплоть до 1868 год

Отчёт JP→RU агента
Время: 18.98 с
LLM-запросы: 2, Wiki-запросы: 1

СПРАВКА ДЛЯ ПЕРЕВОДЧИКА:

1) Самый удачный вариант заголовка для статьи "Сёгунат Токугава" на русском языке будет: "Одноименный сёгунат Токугава".
2) Дословный перевод японского термина "江戸幕府" на русский язык будет: "Японское военное правительство".
3) Краткая справка об этом термине: "江戸幕府" (Японское военное правительство) представляет собой название феодального военного правительства Японии, созданного в 1603 году Токугавой Иэясу. Это правительство было о

In [5]:
import wikipedia
wikipedia.set_lang("ja")
print(wikipedia.search("船手頭"))


['船手頭', '江戸幕府', '牧野成賢', '小浜光隆', '長州藩', '土屋知貞', '東蓮寺藩', '小浜嘉隆', '幕府海軍', '村上水軍']


In [10]:
### 6. Демо-запуск
if __name__ == "__main__":
    sample = "В тексте встретилось 長州藩. Как правильно перевести?"
    print(translate_ja_term(sample))

ЗАПРОС ПЕРЕВОДЧИКА: В тексте встретилось 長州藩. Как правильно перевести?

Найден термин: 長州藩
Найдена русская статья: Тёсю (княжество)
Текст статьи (первые 300 символов): Княжество Тёсю (яп. 長州藩 Тё:сю:-хан) — феодальное княжество (хан) в Японии периода Эдо (1603—1867), занимавшее территорию современной префектуры Ямагути и сыгравшее значительную роль в падении сёгуната Токугава. Также известно как княжество Хаги (яп. 萩藩 Хаги-хан), Ямагути (яп. 山口藩 Ямагути-хан) или Су

Отчёт JP→RU агента
Время: 16.73 с
LLM-запросы: 2, Wiki-запросы: 1

СПРАВКА ДЛЯ ПЕРЕВОДЧИКА:

Для того чтобы найти наиболее подходящий заголовок для статьи "Тёсю" на русском языке, нужно обратиться к русскоязычным источникам, которые могут содержать информацию о данном княжестве. Однако, поскольку нет конкретных источников, доступных мне, я могу только предложить несколько вариантов:

1. **Тёсю** (княжество)
2. **Хаги-хан**
3. **Ямагути**

Однако, если вы хотите получить более точное название, которое соответствует всем данны

In [11]:
### 6. Демо-запуск
if __name__ == "__main__":
    sample = "В тексте встретилось 牧野成賢. Как правильно перевести?"
    print(translate_ja_term(sample))

ЗАПРОС ПЕРЕВОДЧИКА: В тексте встретилось 牧野成賢. Как правильно перевести?

Найден термин: 牧野成賢
❌ Русская интер-вики ссылка не найдена
Русская статья не найдена, попробуйте переформулировать.


In [4]:
import gradio as gr
from IPython.display import Markdown  # только если хотите красивость в colab


def translate_ja_term_gradio(ja_query: str) -> str:
    """
    Обёртка над функцией:
    возвращает только текст справки, без вывода в консоль.
    """
    import time, re, json, wikipedia, requests
    from typing import Dict

    monitor.reset()
    monitor.start = time.time()

    # 1) извлечение термина
    extract_prompt = f"""<|im_start|>system
Ты извлекаешь японский термин из текста пользователя.
1. Ответь **точно теми же иероглифами / каной**, что есть во входе.
2. Не переводи, не транскрибируй, не добавляй пояснений.
3. Выведи **одну непрерывную строку** только из японских символов (кандзи, хирагана, катакана).
4. Если таких символов несколько — верни самую длинную непрерывную последовательность.

Примеры:
Вход: 船手頭是什么？ → 船手頭
Вход: あの人は戦国時代が好きだ → 戦国時代
<|im_start|>user
{ja_query}
<|im_start|>assistant"""
    term_ja = pipe(extract_prompt)[0]["generated_text"].strip()

    # 2) поиск статьи
    ru_title, ru_text = ja2ru_wiki_page(term_ja)
    if not ru_title:
        return "Русская статья не найдена, попробуйте переформулировать."

    # 3) генерация ответа
    final_prompt = f"""<|im_start|>system
Ты — помощник переводчика с японского на русский. Используя факты из русской Википедии (ниже), дай:
1) сообщи, что самый удачный вариант перевода на русский это - заголовок аналогичной статьи в русской Википедии: {ru_title};
2) дай дословный перевод японского термина {term_ja} на русский;
3) дай краткую справку (1-2 предложения) о том, что это такое, используя факты из статьи.

Факты из Википедии:
{ru_text[:1_200]}
<|im_start|>user
Как перевести «{term_ja}»?<|im_start|>assistant"""
    answer = pipe(final_prompt)[0]["generated_text"].strip()

    monitor.end = time.time()
    answer += f"\n\n---\nВремя: {monitor.end-monitor.start:.2f} с, LLM-запросы: {monitor.llm_calls}, Wiki-запросы: {monitor.wiki_calls}"
    return answer


# ------------------------------------------------------------------
# 2. Gradio-интерфейс
# ------------------------------------------------------------------
demo = gr.Interface(
    fn=translate_ja_term_gradio,
    inputs=gr.Textbox(label="Введите фразу с японским термином",
                     placeholder="例: В тексте встретилось 江戸幕府. Как перевести?"),
    outputs=gr.Markdown(label="Справка для переводчика"),
    title="JP-RU Historical Terms Helper",
    description="Интеллектуальный глоссарий для япониста. Вставьте предложение, агент найдёт термин, "
               "вытащит русскую викистатью и подскажет перевод.",
    examples=[
        ["В тексте встретилось 江戸幕府. Как правильно перевести?"],
        ["В тексте встретилось 長州藩. Как правильно перевести?"],
        ["В тексте встретилось 幕府海軍. Как правильно перевести?"],
    ],
    cache_examples=False,
)

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


❌ Русская интер-вики ссылка не найдена
